# TF - Teoria de Compiladores
### Grupo: 4

### Integrantes
- **Diego Fabrizio Mucha Alvarez - u202317069**
- **Nicole Yessenia Vasquez Tinco - u202322884**
- **Alessandro Daniel Bravo Castillo - u202224501**
- **Linda Libertad Leon Quispe - u202322089**
- **Alessandro Elías Hesse Pulache - u202318347**

## 1. Instalación de Dependencias Iniciales

In [ ]:
!pip install antlr4-python3-runtime==4.13.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.9.3
    Uninstalling antlr4-python3-runtime-4.9.3:
      Successfully uninstalled antlr4-python3-runtime-4.9.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.13.2 which is incompatible.


In [ ]:
!curl -O https://www.antlr.org/download/antlr-4.13.2-complete.jar

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2089k  100 2089k    0     0  3755k      0 --:--:-- --:--:-- --:--:-- 3758k


## 2. Gramática en ANTLR4

In [ ]:
%%writefile Formas.g4
grammar Formas;

programa: instrucciones EOF;

instrucciones: instruccion*;

instruccion
    : asignacion
    | punto
    | recta
    | triangulo
    | cuadrado
    | pentagono
    | repetir
    | trasladar
    | mostrar
    | mostrarDetallado
    | mostrarAngulos
    | circulo
    ;

asignacion: ID IGUAL expr ;

repetir: REPETIR expr VECES LPAREN instruccion+ RPAREN ;

punto: PUNTO ID LPAREN expr COMA expr RPAREN colorOpcional ;

recta
    : RECTA ID LPAREN ID COMA ID RPAREN colorOpcional
    | RECTA ID LPAREN expr COMA expr COMA expr COMA expr RPAREN colorOpcional
    ;

triangulo
    : TRIANGULO ID LPAREN ID COMA ID COMA ID RPAREN colorOpcional
    | TRIANGULO ID LPAREN expr COMA expr COMA expr COMA expr COMA expr COMA expr RPAREN colorOpcional
    ;

cuadrado
    : CUADRADO ID LPAREN ID COMA ID COMA ID COMA ID RPAREN colorOpcional
    | CUADRADO ID LPAREN expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr RPAREN colorOpcional
    ;

pentagono
    : PENTAGONO ID LPAREN ID COMA ID COMA ID COMA ID COMA ID RPAREN colorOpcional
    | PENTAGONO ID LPAREN expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr RPAREN colorOpcional
    ;

circulo
    : CIRCULO ID LPAREN ID COMA expr RPAREN colorOpcional
    | CIRCULO ID LPAREN expr COMA expr (COMA expr)? RPAREN colorOpcional
    ;

colorOpcional
    : COLOR colorValor
    |
    ;

colorValor
    : ROJO
    | VERDE
    | AZUL
    | AMARILLO
    | NARANJA
    | MORADO
    | NEGRO
    | MARRON
    ;

trasladar
    : TRASLADAR LPAREN ID COMA expr COMA expr RPAREN
    ;

mostrar
    : MOSTRAR LPAREN ID RPAREN
    | MOSTRAR LPAREN ID DOT ID LPAREN NUM RPAREN RPAREN
    ;

mostrarDetallado
    : MOSTRAR_DETALLADO LPAREN ID RPAREN
    ;

mostrarAngulos
    : MOSTRAR_ANGULOS LPAREN ID RPAREN
    ;

expr
    : expr op=(MULT | DIV) expr
    | expr op=(MAS | MENOS) expr
    | NUM
    | ID
    | LPAREN expr RPAREN
    ;

REPETIR   : 'repetir';
VECES     : 'veces';
PUNTO     : 'punto';
RECTA     : 'recta';

TRIANGULO : 'triangulo';
CUADRADO  : 'cuadrado';
PENTAGONO : 'pentagono';
CIRCULO   : 'circulo';
TRASLADAR : 'trasladar';
MOSTRAR_ANGULOS : 'mostrar_angulos';
MOSTRAR_DETALLADO : 'mostrar_detallado';
MOSTRAR   : 'mostrar';
COLOR     : 'color';
ROJO      : 'rojo';
VERDE     : 'verde';
AZUL      : 'azul';
AMARILLO  : 'amarillo';
NARANJA   : 'naranja';
MORADO    : 'morado';
NEGRO     : 'negro';
MARRON    : 'marron';

IGUAL : '=';
DOT   : '.';
COMA  : ',';
LPAREN: '(';
RPAREN: ')';

MAS   : '+';
MENOS : '-';
MULT  : '*';
DIV   : '/';

ID  : [a-zA-Z][a-zA-Z0-9_]*;
NUM : [0-9]+ ('.' [0-9]+)?;

WS : [ \t\r\n]+ -> skip;

Writing Formas.g4


In [ ]:
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 -visitor Formas.g4

## 3. Desarrollo del Visitor

In [ ]:
%%writefile Visitor.py

import math
if __name__ is not None and "." in __name__:
    from .FormasParser import FormasParser
    from .FormasVisitor import FormasVisitor
else:
    from FormasParser import FormasParser
    from FormasVisitor import FormasVisitor


class Visitor(FormasVisitor):
    COLOR_DEFAULT = "black"
    COLORES_VALIDOS = {
        "rojo": "red",
        "verde": "green",
        "azul": "blue",
        "amarillo": "yellow",
        "naranja": "orange",
        "morado": "purple",
        "negro": "black",
        "marron": "brown",
    }

    def __init__(self):
        self.variables = {}
        self.figuras = {}
        self.html = ""

    ### GET FIGURAS ###
    def getFiguras(self):
        return self.figuras

    ### GET HTML ###
    def getHtml(self):
        return self.html

    ### OBTENER COLOR ###
    def obtenerColor(self, ctx):
        color = ctx.colorOpcional()
        if color and color.colorValor():
            valor = color.colorValor().getText()
            if valor not in self.COLORES_VALIDOS:
                colores = ", ".join(sorted(self.COLORES_VALIDOS))
                raise Exception(f"Color no valido: {valor}. Colores permitidos: {colores}")
            return self.COLORES_VALIDOS[valor]
        return self.COLOR_DEFAULT

    ### FORMATEAR NUMERO ###
    def formatearNumero(self, valor):
        if isinstance(valor, float) and valor.is_integer():
            return str(int(valor))
        if isinstance(valor, float):
            return f"{valor:.2f}"
        return str(valor)

    ### ESCAPAR TEXTO JS ###
    def escaparTexto(self, texto):
        return texto.replace("\\", "\\\\").replace('"', '\\"')

    ### OBTENER PUNTOS DE FIGURA ###
    def obtenerPuntosFigura(self, figura):
        if figura["tipo"] == "punto":
            return [(figura["x"], figura["y"])]
        if figura["tipo"] == "recta":
            return [(figura["x1"], figura["y1"]), (figura["x2"], figura["y2"])]
        if figura["tipo"] in ["triangulo", "cuadrado", "pentagono"]:
            return figura["puntos"]
        if figura["tipo"] == "circulo":
            return [(figura["x"], figura["y"])]
        return []

    ### CALCULAR LADOS ###
    def calcularLados(self, puntos, cerrar=True):
        lados = []
        limite = len(puntos) if cerrar else len(puntos) - 1
        for i in range(limite):
            x1, y1 = puntos[i]
            x2, y2 = puntos[(i + 1) % len(puntos)]
            lados.append(math.dist((x1, y1), (x2, y2)))
        return lados

    ### GENERAR ETIQUETAS DE LADOS ###
    def generarDetalleFigura(self, nombre, figura):
        tipo = figura["tipo"]
        if tipo == "recta":
            puntos = self.obtenerPuntosFigura(figura)
            cerrar = False
        elif tipo in ["triangulo", "cuadrado", "pentagono"]:
            puntos = self.obtenerPuntosFigura(figura)
            cerrar = True
        else:
            return ""

        codigo = """
            ctx.save();
            ctx.fillStyle = "black";
            ctx.font = "12px Arial";
        """

        limite = len(puntos) if cerrar else len(puntos) - 1
        for i in range(limite):
            x1, y1 = puntos[i]
            x2, y2 = puntos[(i + 1) % len(puntos)]
            longitud = math.dist((x1, y1), (x2, y2))
            etiqueta = f"{self.formatearNumero(longitud)} m"
            x_medio = (x1 + x2) / 2
            y_medio = (y1 + y2) / 2
            codigo += f"""
            ctx.fillText("{self.escaparTexto(etiqueta)}", {x_medio} + 5, {y_medio} - 5);
            """

        codigo += """
            ctx.restore();
        """

        return codigo

    ### CALCULAR ANGULO INTERNO ###
    def calcularAnguloInterno(self, anterior, actual, siguiente):
        ax, ay = anterior
        bx, by = actual
        cx, cy = siguiente

        vector1 = (ax - bx, ay - by)
        vector2 = (cx - bx, cy - by)
        magnitud1 = math.hypot(vector1[0], vector1[1])
        magnitud2 = math.hypot(vector2[0], vector2[1])

        if magnitud1 == 0 or magnitud2 == 0:
            return 0

        producto = vector1[0] * vector2[0] + vector1[1] * vector2[1]
        coseno = max(-1, min(1, producto / (magnitud1 * magnitud2)))
        return math.degrees(math.acos(coseno))

    ### CALCULAR MARCA DE ANGULO ###
    def calcularMarcaAngulo(self, anterior, actual, siguiente, centro):
        ax, ay = anterior
        bx, by = actual
        cx, cy = siguiente
        centro_x, centro_y = centro

        longitud1 = math.dist(actual, anterior)
        longitud2 = math.dist(actual, siguiente)
        radio = max(14, min(30, longitud1 * 0.22, longitud2 * 0.22))

        inicio = math.atan2(ay - by, ax - bx)
        fin = math.atan2(cy - by, cx - bx)
        vuelta = 2 * math.pi
        delta_ccw = (fin - inicio) % vuelta

        medio_ccw = inicio + delta_ccw / 2
        medio_cw = inicio - (vuelta - delta_ccw) / 2

        punto_ccw = (bx + math.cos(medio_ccw) * radio, by + math.sin(medio_ccw) * radio)
        punto_cw = (bx + math.cos(medio_cw) * radio, by + math.sin(medio_cw) * radio)
        distancia_ccw = math.dist(punto_ccw, (centro_x, centro_y))
        distancia_cw = math.dist(punto_cw, (centro_x, centro_y))

        if distancia_ccw <= distancia_cw:
            return inicio, fin, False, medio_ccw, radio
        return inicio, fin, True, medio_cw, radio

    ### GENERAR ETIQUETAS DE ANGULOS ###
    def generarAngulosFigura(self, nombre, figura):
        if figura["tipo"] not in ["triangulo", "cuadrado", "pentagono"]:
            raise Exception("mostrar_angulos solo se puede usar con triangulo, cuadrado o pentagono")

        puntos = figura["puntos"]
        centro_x = sum(x for x, _ in puntos) / len(puntos)
        centro_y = sum(y for _, y in puntos) / len(puntos)

        codigo = """
            ctx.save();
            ctx.strokeStyle = "black";
            ctx.fillStyle = "black";
            ctx.font = "12px Arial";
            ctx.lineWidth = 1.5;
        """

        for i, punto in enumerate(puntos):
            anterior = puntos[i - 1]
            siguiente = puntos[(i + 1) % len(puntos)]
            angulo = self.calcularAnguloInterno(anterior, punto, siguiente)
            inicio, fin, antihorario, direccion_texto, radio = self.calcularMarcaAngulo(
                anterior,
                punto,
                siguiente,
                (centro_x, centro_y)
            )
            x, y = punto
            x_texto = x + math.cos(direccion_texto) * (radio + 10)
            y_texto = y + math.sin(direccion_texto) * (radio + 10)
            etiqueta = f"{self.formatearNumero(angulo)} grados"
            codigo += f"""
            ctx.beginPath();
            ctx.arc({x}, {y}, {radio}, {inicio}, {fin}, {str(antihorario).lower()});
            ctx.stroke();
            ctx.fillText("{self.escaparTexto(etiqueta)}", {x_texto}, {y_texto});
            """

        codigo += """
            ctx.restore();
        """

        return codigo

    ### CALCULAR LINEA NOTABLE
    def calcularLineaNotable(self, triangulo, tipo_linea):
        import math

        puntos = triangulo["puntos"]
        lineas = []

        for i in range(3):
            x1, y1 = puntos[i]
            x2, y2 = puntos[(i + 1) % 3]
            x3, y3 = puntos[(i + 2) % 3]

            if tipo_linea == "mediana":
                px = (x2 + x3) / 2
                py = (y2 + y3) / 2

            elif tipo_linea == "altura":
                dx = x3 - x2
                dy = y3 - y2
                t = ((x1 - x2) * dx + (y1 - y2) * dy) / (dx * dx + dy * dy)
                px = x2 + t * dx
                py = y2 + t * dy

            elif tipo_linea == "bisectriz":
                lado1 = math.dist((x1, y1), (x2, y2))
                lado2 = math.dist((x1, y1), (x3, y3))
                px = (lado2 * x2 + lado1 * x3) / (lado1 + lado2)
                py = (lado2 * y2 + lado1 * y3) / (lado1 + lado2)

            lineas.append([x1, y1, px, py])

        return lineas

    ### VISIT PROGRAMA ###
    def visitPrograma(self, ctx):
        return self.visit(ctx.instrucciones())

    ### VISIT INSTRUCCIONES ###
    def visitInstrucciones(self, ctx):
        for instruccion in ctx.instruccion():
            self.visit(instruccion)

    ### VISIT REPETIR ###
    def visitRepetir(self, ctx):
        num = self.visit(ctx.expr())
        for i in range(num):
            for instruccion in ctx.instruccion():
                self.visit(instruccion)

    ### VISIT ASIGNACION ###
    def visitAsignacion(self, ctx):
        nombre = ctx.ID().getText()
        valor = self.visit(ctx.expr())
        self.variables[nombre] = valor

    ### VISIT PUNTO ###
    def visitPunto(self, ctx):
        nombre = ctx.ID().getText()
        x = self.visit(ctx.expr(0))
        y = self.visit(ctx.expr(1))
        self.figuras[nombre] = {
            "tipo": "punto",
            "x": x,
            "y": y,
            "color": self.obtenerColor(ctx)
        }

    ### VISIT RECTA ###
    def visitRecta(self, ctx):
        nombre = ctx.ID(0).getText()
        color = self.obtenerColor(ctx)
        if len(ctx.ID()) == 3:
            p1 = ctx.ID(1).getText()
            p2 = ctx.ID(2).getText()
            punto1 = self.figuras[p1]
            punto2 = self.figuras[p2]
            self.figuras[nombre] = {
                "tipo": "recta",
                "x1": punto1["x"],
                "y1": punto1["y"],
                "x2": punto2["x"],
                "y2": punto2["y"],
                "color": color
            }
        else:
            self.figuras[nombre] = {
                "tipo": "recta",
                "x1": self.visit(ctx.expr(0)),
                "y1": self.visit(ctx.expr(1)),
                "x2": self.visit(ctx.expr(2)),
                "y2": self.visit(ctx.expr(3)),
                "color": color
            }

    ### VISIT TRIANGULO ###
    def visitTriangulo(self, ctx):
        nombre = ctx.ID(0).getText()
        color = self.obtenerColor(ctx)
        if len(ctx.ID()) == 4:
            p1 = self.figuras[ctx.ID(1).getText()]
            p2 = self.figuras[ctx.ID(2).getText()]
            p3 = self.figuras[ctx.ID(3).getText()]
            self.figuras[nombre] = {
                "tipo": "triangulo",
                "puntos": [
                    (p1["x"], p1["y"]),
                    (p2["x"], p2["y"]),
                    (p3["x"], p3["y"])
                ],
                "color": color
            }
        else:
            self.figuras[nombre] = {
                "tipo": "triangulo",
                "puntos": [
                    (self.visit(ctx.expr(0)), self.visit(ctx.expr(1))),
                    (self.visit(ctx.expr(2)), self.visit(ctx.expr(3))),
                    (self.visit(ctx.expr(4)), self.visit(ctx.expr(5)))
                ],
                "color": color
            }
    ### VISIT CUADRADO ###
    def visitCuadrado(self, ctx):
        nombre = ctx.ID(0).getText()
        color = self.obtenerColor(ctx)
        if len(ctx.ID()) == 5:
            p1 = self.figuras[ctx.ID(1).getText()]
            p2 = self.figuras[ctx.ID(2).getText()]
            p3 = self.figuras[ctx.ID(3).getText()]
            p4 = self.figuras[ctx.ID(4).getText()]
            self.figuras[nombre] = {
                "tipo": "cuadrado",
                "puntos": [
                    (p1["x"], p1["y"]),
                    (p2["x"], p2["y"]),
                    (p3["x"], p3["y"]),
                    (p4["x"], p4["y"])
                ],
                "color": color
            }
        else:
            self.figuras[nombre] = {
                "tipo": "cuadrado",
                "puntos": [
                    (self.visit(ctx.expr(0)), self.visit(ctx.expr(1))),
                    (self.visit(ctx.expr(2)), self.visit(ctx.expr(3))),
                    (self.visit(ctx.expr(4)), self.visit(ctx.expr(5))),
                    (self.visit(ctx.expr(6)), self.visit(ctx.expr(7)))
                ],
                "color": color
            }

    ### VISIT CIRCULO ###
    def visitCirculo(self, ctx):
        nombre = ctx.ID(0).getText()
        color = self.obtenerColor(ctx)

        if len(ctx.ID()) == 2:
            punto_id = ctx.ID(1).getText()
            centro = self.figuras[punto_id]
            x = centro["x"]
            y = centro["y"]
            radio = self.visit(ctx.expr(0))

        else:
            exprs = ctx.expr()
            x = self.visit(exprs[0])
            y = self.visit(exprs[1])
            radio = self.visit(exprs[2]) if len(exprs) == 3 else 30

        self.figuras[nombre] = {
            "tipo": "circulo",
            "x": x,
            "y": y,
            "radio": radio,
            "color": color
        }

    ### VISIT PENTAGONO ###
    def visitPentagono(self, ctx):
        nombre = ctx.ID(0).getText()
        color = self.obtenerColor(ctx)

        if len(ctx.ID()) == 6:
            p1 = self.figuras[ctx.ID(1).getText()]
            p2 = self.figuras[ctx.ID(2).getText()]
            p3 = self.figuras[ctx.ID(3).getText()]
            p4 = self.figuras[ctx.ID(4).getText()]
            p5 = self.figuras[ctx.ID(5).getText()]

            self.figuras[nombre] = {
                "tipo": "pentagono",
                "puntos": [
                    (p1["x"], p1["y"]),
                    (p2["x"], p2["y"]),
                    (p3["x"], p3["y"]),
                    (p4["x"], p4["y"]),
                    (p5["x"], p5["y"])
                ],
                "color": color
            }

        else:
            self.figuras[nombre] = {
                "tipo": "pentagono",
                "puntos": [
                    (self.visit(ctx.expr(0)), self.visit(ctx.expr(1))),
                    (self.visit(ctx.expr(2)), self.visit(ctx.expr(3))),
                    (self.visit(ctx.expr(4)), self.visit(ctx.expr(5))),
                    (self.visit(ctx.expr(6)), self.visit(ctx.expr(7))),
                    (self.visit(ctx.expr(8)), self.visit(ctx.expr(9)))
                ],
                "color": color
            }
    ### VISIT TRASLADAR ###
    def visitTrasladar(self, ctx):
        nombre = ctx.ID().getText()
        dx = self.visit(ctx.expr(0))
        dy = self.visit(ctx.expr(1))
        figura = self.figuras[nombre]

        if figura["tipo"] in ["punto", "circulo"]:
            figura["x"] += dx
            figura["y"] += dy
        elif figura["tipo"] == "recta":
            figura["x1"] += dx; figura["y1"] += dy
            figura["x2"] += dx; figura["y2"] += dy
        elif figura["tipo"] in ["triangulo", "cuadrado", "pentagono"]:
            nuevos = []
            for x, y in figura["puntos"]:
                nuevos.append((x + dx, y + dy))
            figura["puntos"] = nuevos

    ### VISIT MOSTRAR ###
    def visitMostrar(self, ctx):
        if len(ctx.ID()) == 1:
            nombre = ctx.ID(0).getText()
            figura = self.figuras[nombre]
            self.html += self.generarFigura(nombre, figura)

        elif len(ctx.ID()) == 2:
            nombre = ctx.ID(0).getText()
            figura = self.figuras[nombre]
            linea_notable = ctx.ID(1).getText()
            lineas = self.calcularLineaNotable(figura, linea_notable)
            figura[linea_notable] = lineas
            self.html += self.generarFigura(nombre, figura, int(ctx.NUM().getText()))

    ### VISIT MOSTRAR DETALLADO ###
    def visitMostrarDetallado(self, ctx):
        nombre = ctx.ID().getText()
        figura = self.figuras[nombre]
        self.html += self.generarFigura(nombre, figura)
        self.html += self.generarDetalleFigura(nombre, figura)

    ### VISIT MOSTRAR ANGULOS ###
    def visitMostrarAngulos(self, ctx):
        nombre = ctx.ID().getText()
        figura = self.figuras[nombre]
        self.html += self.generarFigura(nombre, figura)
        self.html += self.generarAngulosFigura(nombre, figura)

    ### GENERAR FIGURA ###
    def generarFigura(self, nombre, figura, id_ln=0):
        codigo = ""
        color = figura.get("color", self.COLOR_DEFAULT)
        if figura["tipo"] == "punto":
            x = figura["x"]
            y = figura["y"]
            codigo += f"""
            ctx.strokeStyle = "{color}";
            ctx.fillStyle = "{color}";
            ctx.beginPath();
            ctx.arc({x}, {y}, 5, 0, 2 * Math.PI);
            ctx.fill();
            ctx.fillText("{nombre}({x},{y})", {x}+10, {y}+10);
            """

        elif figura["tipo"] == "recta":
            x1 = figura["x1"]
            y1 = figura["y1"]
            x2 = figura["x2"]
            y2 = figura["y2"]
            codigo += f"""
            ctx.strokeStyle = "{color}";
            ctx.fillStyle = "{color}";
            ctx.beginPath();
            ctx.moveTo({x1}, {y1});
            ctx.lineTo({x2}, {y2});
            ctx.stroke();
            ctx.fillText("{nombre}", ({x1}+{x2})/2, ({y1}+{y2})/2 - 12);
            """

        elif figura["tipo"] == "triangulo":
            for tipo in ["mediana", "bisectriz", "altura"]:
                if tipo in figura and id_ln > 0:
                    ln = figura[tipo][id_ln - 1]
                    codigo += f"""
                    ctx.strokeStyle = "{color}";
                    ctx.fillStyle = "{color}";
                    ctx.beginPath();
                    ctx.moveTo({ln[0]}, {ln[1]});
                    ctx.lineTo({ln[2]}, {ln[3]});
                    ctx.stroke();
                    ctx.fillText("{tipo}", {ln[2]}, {ln[3]});
                    """

            p = figura["puntos"]
            codigo += f"""
            ctx.strokeStyle = "{color}";
            ctx.fillStyle = "{color}";
            ctx.beginPath();
            ctx.moveTo({p[0][0]}, {p[0][1]});
            ctx.lineTo({p[1][0]}, {p[1][1]});
            ctx.lineTo({p[2][0]}, {p[2][1]});
            ctx.closePath();
            ctx.stroke();
            ctx.fillText(
                "{nombre}",
                ({p[0][0]} + {p[1][0]} + {p[2][0]}) / 3,
                ({p[0][1]} + {p[1][1]} + {p[2][1]}) / 3
            );
            """

        elif figura["tipo"] == "cuadrado":
            p = figura["puntos"]
            codigo += f"""
            ctx.strokeStyle = "{color}";
            ctx.fillStyle = "{color}";
            ctx.beginPath();
            ctx.moveTo({p[0][0]}, {p[0][1]});
            ctx.lineTo({p[1][0]}, {p[1][1]});
            ctx.lineTo({p[2][0]}, {p[2][1]});
            ctx.lineTo({p[3][0]}, {p[3][1]});
            ctx.closePath();
            ctx.stroke();
            ctx.fillText(
                "{nombre}",
                ({p[0][0]} + {p[1][0]} + {p[2][0]} + {p[3][0]}) / 4,
                ({p[0][1]} + {p[1][1]} + {p[2][1]} + {p[3][1]}) / 4
            );
            """

        elif figura["tipo"] == "circulo":
            x = figura["x"]
            y = figura["y"]
            r = figura["radio"]
            codigo += f"""
            ctx.strokeStyle = "{color}";
            ctx.fillStyle = "{color}";
            ctx.beginPath();
            ctx.arc({x}, {y}, {r}, 0, 2 * Math.PI);
            ctx.stroke();
            ctx.fillText("{nombre}", {x} + {r} + 5, {y});
            """

        elif figura["tipo"] == "pentagono":
            p = figura["puntos"]

            codigo += f"""
            ctx.strokeStyle = "{color}";
            ctx.fillStyle = "{color}";
            ctx.beginPath();
            ctx.moveTo({p[0][0]}, {p[0][1]});
            ctx.lineTo({p[1][0]}, {p[1][1]});
            ctx.lineTo({p[2][0]}, {p[2][1]});
            ctx.lineTo({p[3][0]}, {p[3][1]});
            ctx.lineTo({p[4][0]}, {p[4][1]});
            ctx.closePath();
            ctx.stroke();

            ctx.fillText(
                "{nombre}",
                ({p[0][0]} + {p[1][0]} + {p[2][0]} + {p[3][0]} + {p[4][0]}) / 5,
                ({p[0][1]} + {p[1][1]} + {p[2][1]} + {p[3][1]} + {p[4][1]}) / 5
            );
            """

        return codigo

    ### VISIT EXPR ###
    def visitExpr(self, ctx):
        if ctx.NUM():
            n = ctx.NUM().getText()
            return float(n) if "." in n else int(n)

        if ctx.ID():
            nombre = ctx.ID().getText()
            return self.variables[nombre]

        if ctx.getChildCount() == 3:
            if ctx.getChild(0).getText() == "(":
                return self.visit(ctx.expr(0))

            izquierda = self.visit(ctx.expr(0))
            derecha = self.visit(ctx.expr(1))
            operador = ctx.getChild(1).getText()

            if operador == "+":
                return izquierda + derecha
            elif operador == "-":
                return izquierda - derecha
            elif operador == "*":
                return izquierda * derecha
            elif operador == "/":
                return izquierda / derecha

Writing Visitor.py


## 4. Ejemplos inciales de ejecución

In [ ]:
%%writefile ejemplo1.geo

lado = 120
radio = 45

punto A(120, 120) color rojo
punto B(240, 120) color azul
punto C(180, 240) color verde

triangulo T1(A, B, C) color morado

mostrar_detallado(A)
mostrar(B)
mostrar(C)
mostrar_angulos(T1)

circulo C1(A, 20) color naranja
circulo C2(B, 20) color amarillo
circulo C3(C, 20) color marron

mostrar(C1)
mostrar(C2)
mostrar(C3)

punto D(420, 100) color rojo
punto E(560, 100) color azul
punto F(560, 240) color verde
punto G(420, 240) color morado

cuadrado Q1(D, E, F, G) color naranja

mostrar(D)
mostrar(E)
mostrar(F)
mostrar(G)

mostrar_detallado(Q1)

recta R1(D, F) color amarillo
recta R2(E, G) color marron

mostrar(R1)
mostrar(R2)

punto H(760, 110) color rojo
punto I(860, 170) color azul
punto J(830, 290) color verde
punto K(690, 290) color morado
punto L(660, 170) color naranja

pentagono P1(H, I, J, K, L) color marron


mostrar_angulos(P1)

circulo Sol(980, 120) color amarillo

mostrar(Sol)

repetir 6 veces (
    trasladar(Sol, -15, 15)
    mostrar(Sol)
)

recta Camino1(100, 420, 300, 420) color rojo
recta Camino2(300, 420, 500, 500) color verde
recta Camino3(500, 500, 750, 420) color azul

mostrar_detallado(Camino1)
mostrar(Camino2)
mostrar(Camino3)

punto X1(880, 420) color negro
punto X2(980, 420) color negro
punto X3(980, 520) color negro
punto X4(880, 520) color negro

cuadrado Casa(X1, X2, X3, X4) color marron

mostrar_detallado(Casa)

triangulo Techo(880, 420, 980, 420, 930, 340) color naranja

mostrar_detallado(Techo)

circulo Ventana(930, 470, 20) color azul

mostrar_detallado(Ventana)

Writing ejemplo1.geo


In [ ]:
%%writefile ejemplo2.geo

punto A1(120, 120)
punto B1(300, 120)
punto C1(210, 300)

triangulo TMed(A1, B1, C1) color rojo

mostrar(TMed)
mostrar(TMed.mediana(1))
mostrar(TMed.mediana(2))
mostrar(TMed.mediana(3))

punto A2(450, 100)
punto B2(700, 160)
punto C2(520, 320)

triangulo TAlt(A2, B2, C2) color verde

mostrar(TAlt)
mostrar(TAlt.altura(1))
mostrar(TAlt.altura(2))
mostrar(TAlt.altura(3))

punto A3(850, 120)
punto B3(1080, 180)
punto C3(920, 360)

triangulo TBis(A3, B3, C3) color azul

mostrar(TBis)
mostrar(TBis.bisectriz(1))
mostrar(TBis.bisectriz(2))
mostrar(TBis.bisectriz(3))

Overwriting ejemplo2.geo


In [ ]:
from antlr4 import *
import importlib
import FormasLexer as FormasLexerModule
import FormasParser as FormasParserModule
import Visitor as VisitorModule

importlib.reload(FormasLexerModule)
importlib.reload(FormasParserModule)
importlib.reload(VisitorModule)

FormasLexer = FormasLexerModule.FormasLexer
FormasParser = FormasParserModule.FormasParser
Visitor = VisitorModule.Visitor

input_stream = FileStream('ejemplo2.geo')

lexer = FormasLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = FormasParser(token_stream)

tree = parser.programa()

print(tree.toStringTree(recog=parser))

visitor = Visitor()
visitor.visit(tree)

print(visitor.getFiguras())

(programa (instrucciones (instruccion (punto punto A1 ( (expr 120) , (expr 120) ) colorOpcional)) (instruccion (punto punto B1 ( (expr 300) , (expr 120) ) colorOpcional)) (instruccion (punto punto C1 ( (expr 210) , (expr 300) ) colorOpcional)) (instruccion (triangulo triangulo TMed ( A1 , B1 , C1 ) (colorOpcional color (colorValor rojo)))) (instruccion (mostrar mostrar ( TMed ))) (instruccion (mostrar mostrar ( TMed . mediana ( 1 ) ))) (instruccion (mostrar mostrar ( TMed . mediana ( 2 ) ))) (instruccion (mostrar mostrar ( TMed . mediana ( 3 ) ))) (instruccion (punto punto A2 ( (expr 450) , (expr 100) ) colorOpcional)) (instruccion (punto punto B2 ( (expr 700) , (expr 160) ) colorOpcional)) (instruccion (punto punto C2 ( (expr 520) , (expr 320) ) colorOpcional)) (instruccion (triangulo triangulo TAlt ( A2 , B2 , C2 ) (colorOpcional color (colorValor verde)))) (instruccion (mostrar mostrar ( TAlt ))) (instruccion (mostrar mostrar ( TAlt . altura ( 1 ) ))) (instruccion (mostrar mostrar (

In [ ]:
from IPython.display import HTML

html_content = f"""
<div style="padding: 20px; background-color: #f0f0f0; border-radius: 10px;">
  <h2 style="color: #333;">Interfaz de figuras geométricas</h2>

  <canvas id="canvas" width="1200" height="700" style="background-color: white; border: 1px solid #999;"></canvas>

  <script>
    const canvas = document.getElementById("canvas");
    const ctx = canvas.getContext("2d");

    ctx.font = "12px Arial";
    ctx.lineWidth = 2;

    {visitor.getHtml()}
  </script>
</div>
"""

HTML(html_content)

## 5. Pruebas de validación

En esta sección se ejecutan casos correctos incorrectos para evidenciar que el programa tiene un funcionamiento adecuado.

Vamos a presentar 4 casos correctos y 2 incorrectos

In [1]:
!mkdir -p casos_de_exito

In [2]:
!mkdir -p casos_de_error

A subdirectory or file -p already exists.
Error occurred while processing: -p.


### 5.1. Casos de éxito

**Caso 1: Creación de figuras simples (puntos y líneas)**
- **Objetivo:** Validar que el analizador léxico y sintáctico reconoce los tokens fundamentales de posicionamiento y longitud.
- **Entrada de prueba:** Instrucciones de creación para entidades Punto y Línea con coordenadas explícitas.
- **Resultado:** Representación visual de los elementos en las coordenadas exactas de píxeles calculadas por el Visitor.

In [3]:
%%writefile casos_de_exito/caso1.geo
punto A(100, 100)
punto B(300, 250)
punto C(200, 50)
punto D(300, 250)

recta R1(A, B) color verde
recta R2(C, D) color azul

mostrar(R1)
mostrar(R2)

Writing casos_de_exito/caso1.geo


**Caso 2: Creación de figuras complejas (cuadrados y círculos)**
- **Objetivo:** Validar la correcta interpretación de estructuras con parámetros compuestos (coordenadas y radios)
- **Entrada de prueba:** Instrucciones de creación para entidades Cuadrado y Círculo con radio y aristas correspondientes.
- **Resultado:** Representación visual de los elementos (figuras complejas) en las coordenadas exactas de píxeles calculadas por el Visitor.

In [4]:
%%writefile casos_de_exito/caso2.geo
punto A(120, 120) color rojo
punto B(240, 120) color azul
punto C(180, 240) color verde

circulo C1(A, 20) color naranja
circulo C2(B, 20) color amarillo
circulo C3(C, 20) color marron

mostrar(C1)
mostrar(C2)
mostrar(C3)

punto D(420, 100) color rojo
punto E(560, 100) color azul
punto F(560, 240) color verde
punto G(420, 240) color morado

cuadrado Q1(D, E, F, G) color naranja

mostrar(D)
mostrar(E)
mostrar(F)
mostrar(G)

mostrar(Q1)

punto H(760, 110) color rojo
punto I(860, 170) color azul
punto J(830, 290) color verde
punto K(690, 290) color morado
punto L(660, 170) color naranja

pentagono P1(H, I, J, K, L) color marron

Writing casos_de_exito/caso2.geo


**Caso 3: Aplicación de transformaciones lineales (traslación y escalado)**
- **Objetivo:** Comprobar que las matrices de transformación matemática (traslación y escalado) se aplican correctamente sobre las figuras.
- **Entrada de prueba:** Instrucciones de transformación para entidades Cuadrado y Círculo.
- **Resultado:** El sistema actualiza la posición del cuadrado en los ejes X e Y, y redibuja el círculo con un tamaño un 50% mayor al original, manteniendo su centro.

In [5]:
%%writefile casos_de_exito/caso3.geo
lado = 100
radio = 40
radio_escalado = radio * 1.5

punto A(80, 80) color rojo
punto B(80 + lado, 80) color azul
punto C(80 + lado, 80 + lado) color verde
punto D(80, 80 + lado) color morado

cuadrado Q(A, B, C, D) color naranja
mostrar(Q)

trasladar(Q, 220, 80)
mostrar(Q)

circulo CBase(520, 130, radio) color azul
circulo CEscalado(520, 130, radio_escalado) color verde

mostrar(CBase)
mostrar(CEscalado)

Writing casos_de_exito/caso3.geo


**Caso 4: Aplicación de líneas notables para triángulos (altura, bisectriz y mediana)**
- **Objetivo:** Validar la capacidad avanzada del lenguaje para calcular propiedades geométricas complejas (alturas, bisectrices y medianas) de manera automatizada.
- **Entrada de prueba:** Instrucción para el cálculo de la bisectriz de una entidad Triángulo.
- **Resultado:** El sistema calcula y traza la línea notable bisectriz correspondiente sobre el triángulo.


In [6]:
%%writefile casos_de_exito/caso4.geo
punto A1(120, 120)
punto B1(300, 120)
punto C1(210, 300)

triangulo TMed(A1, B1, C1) color rojo

mostrar(TMed)
mostrar(TMed.mediana(1))
mostrar(TMed.mediana(2))
mostrar(TMed.mediana(3))

punto A2(450, 100)
punto B2(700, 160)
punto C2(520, 320)

triangulo TAlt(A2, B2, C2) color azul

mostrar(TAlt)
mostrar(TAlt.altura(1))
mostrar(TAlt.altura(2))
mostrar(TAlt.altura(3))

punto A3(850, 120)
punto B3(1080, 180)
punto C3(920, 360)

triangulo TBis(A3, B3, C3) color verde

mostrar(TBis)
mostrar(TBis.bisectriz(1))
mostrar(TBis.bisectriz(2))
mostrar(TBis.bisectriz(3))

Writing casos_de_exito/caso4.geo


### 5.2. Casos de error

**Caso 1: Renderizar figuras tridimensionales**
- **Objetivo:** Validar que debido al alcance, el lenguaje no tiene la capacidad para entender o renderizar figuras tridimensionales.
- **Entrada de prueba:** Instrucción con el nombre de figuras tridimensionales como: Esfera, Pirámide y Prisma.
- **Resultado:** El sistema busca las figuras tridimensionales en las figuras que puede procesar y no las encuentra, devolviendo un error.

In [7]:
%%writefile casos_de_error/caso1.geo
esfera E(180, 140, 50)
piramide P(360, 240, 80)
prisma PR(520, 180, 100, 60)

mostrar(E)
mostrar(P)
mostrar(PR)

Writing casos_de_error/caso1.geo


**Caso 2: Dibujar una linea notable que no sea las 3 principales**
- **Objetivo:** Validar que debido al alcance, el lenguaje no tiene la capacidad de dibujar o crear otra línea notable que no sean las 3 principales en triángulos que implementamos: Mediana, Bisectriz y Altura.
- **Entrada de prueba:** Instrucción que intenta dibujar `mediatriz`, una línea notable que no está implementada en el visitor.
- **Resultado:** El sistema no reconoce esa línea notable y devuelve un error.

In [8]:
%%writefile casos_de_error/caso2.geo
punto A(120, 120)
punto B(360, 120)
punto C(240, 300)

triangulo T(A, B, C) color morado

mostrar(T.mediatriz(1))

Writing casos_de_error/caso2.geo


### 5.3. Visualizar caso elegido

In [ ]:
from antlr4 import *
from FormasLexer import FormasLexer
from FormasParser import FormasParser
from Visitor import Visitor

input_stream = FileStream('casos_de_exito/caso1.geo')

lexer = FormasLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = FormasParser(token_stream)

tree = parser.programa()

print(tree.toStringTree(recog=parser))

visitor = Visitor()
visitor.visit(tree)

print(visitor.getFiguras())

(programa (instrucciones (instruccion (punto punto A ( (expr 100) , (expr 100) ) colorOpcional)) (instruccion (punto punto B ( (expr 300) , (expr 250) ) colorOpcional)) (instruccion (punto punto C ( (expr 200) , (expr 50) ) colorOpcional)) (instruccion (punto punto D ( (expr 300) , (expr 250) ) colorOpcional)) (instruccion (recta recta R1 ( A , B ) (colorOpcional color (colorValor verde)))) (instruccion (recta recta R2 ( C , D ) (colorOpcional color (colorValor azul)))) (instruccion (mostrar mostrar ( R1 ))) (instruccion (mostrar mostrar ( R2 )))) <EOF>)
{'A': {'tipo': 'punto', 'x': 100, 'y': 100, 'color': 'black'}, 'B': {'tipo': 'punto', 'x': 300, 'y': 250, 'color': 'black'}, 'C': {'tipo': 'punto', 'x': 200, 'y': 50, 'color': 'black'}, 'D': {'tipo': 'punto', 'x': 300, 'y': 250, 'color': 'black'}, 'R1': {'tipo': 'recta', 'x1': 100, 'y1': 100, 'x2': 300, 'y2': 250, 'color': 'green'}, 'R2': {'tipo': 'recta', 'x1': 200, 'y1': 50, 'x2': 300, 'y2': 250, 'color': 'blue'}}


In [ ]:
from IPython.display import HTML

html_content = f"""
<div style="padding: 20px; background-color: #f0f0f0; border-radius: 10px;">
  <h2 style="color: #333;">Prueba de Validación: Caso Error 1</h2>

  <canvas id="canvas_baricentro" width="700" height="500" style="background-color: white; border: 1px solid #999;"></canvas>

  <script>
    const canvas = document.getElementById("canvas_baricentro");
    const ctx = canvas.getContext("2d");

    ctx.font = "12px Arial";
    ctx.lineWidth = 2;

    {visitor.getHtml()}
  </script>
</div>
"""

HTML(html_content)

## 6. Caso con Problema Geométrico Real

### Problema real de cálculo del baricentro de un triángulo

En este caso de prueba se busca demostrar que el lenguaje permite resolver un problema geométrico básico: construir un triángulo a partir de tres puntos, trazar sus medianas e identificar su baricentro.

El baricentro es el punto donde se intersectan las tres medianas de un triángulo. Para hallarlo, se calcula el promedio de las coordenadas de sus tres vértices.

Para un triángulo con vértices:

- A(120, 120)
- B(420, 120)
- C(270, 360)

El baricentro se calcula de la siguiente manera:

$$
G_x = \frac{x_A + x_B + x_C}{3}
$$

$$
G_y = \frac{y_A + y_B + y_C}{3}
$$

Reemplazando los valores:

$$
G_x = \frac{120 + 420 + 270}{3} = 270
$$

$$
G_y = \frac{120 + 120 + 360}{3} = 200
$$

Por lo tanto, el punto baricentro es:

$$
G(270, 200)
$$

El programa debe construir el triángulo, mostrar sus tres medianas y ubicar el punto `G`, que representa el baricentro.

In [ ]:
%%writefile caso_baricentro.geo

punto A(120, 120)
punto B(420, 120)
punto C(270, 360)

triangulo T(A, B, C)

gx = (120 + 420 + 270) / 3
gy = (120 + 120 + 360) / 3

punto G(gx, gy)

mostrar(T)
mostrar(T.mediana(1))
mostrar(T.mediana(2))
mostrar(T.mediana(3))
mostrar(G)

Writing caso_baricentro.geo


In [ ]:
from antlr4 import *
from FormasLexer import FormasLexer
from FormasParser import FormasParser
from Visitor import Visitor

input_stream = FileStream('caso_baricentro.geo')

lexer = FormasLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = FormasParser(token_stream)

tree = parser.programa()

print(tree.toStringTree(recog=parser))

visitor = Visitor()
visitor.visit(tree)

print(visitor.getFiguras())


(programa (instrucciones (instruccion (punto punto A ( (expr 120) , (expr 120) ) colorOpcional)) (instruccion (punto punto B ( (expr 420) , (expr 120) ) colorOpcional)) (instruccion (punto punto C ( (expr 270) , (expr 360) ) colorOpcional)) (instruccion (triangulo triangulo T ( A , B , C ) colorOpcional)) (instruccion (asignacion gx = (expr (expr ( (expr (expr (expr 120) + (expr 420)) + (expr 270)) )) / (expr 3)))) (instruccion (asignacion gy = (expr (expr ( (expr (expr (expr 120) + (expr 120)) + (expr 360)) )) / (expr 3)))) (instruccion (punto punto G ( (expr gx) , (expr gy) ) colorOpcional)) (instruccion (mostrar mostrar ( T ))) (instruccion (mostrar mostrar ( T . mediana ( 1 ) ))) (instruccion (mostrar mostrar ( T . mediana ( 2 ) ))) (instruccion (mostrar mostrar ( T . mediana ( 3 ) ))) (instruccion (mostrar mostrar ( G )))) <EOF>)
{'A': {'tipo': 'punto', 'x': 120, 'y': 120, 'color': 'black'}, 'B': {'tipo': 'punto', 'x': 420, 'y': 120, 'color': 'black'}, 'C': {'tipo': 'punto', 'x': 

In [ ]:
from IPython.display import HTML

html_content = f"""
<div style="padding: 20px; background-color: #f0f0f0; border-radius: 10px;">
  <h2 style="color: #333;">Caso geométrico: baricentro y medianas</h2>

  <canvas id="canvas_baricentro" width="700" height="500" style="background-color: white; border: 1px solid #999;"></canvas>

  <script>
    const canvas = document.getElementById("canvas_baricentro");
    const ctx = canvas.getContext("2d");

    ctx.font = "12px Arial";
    ctx.lineWidth = 2;

    {visitor.getHtml()}
  </script>
</div>
"""

HTML(html_content)